# Profilage DVF 2022 — Etape 2g : ecart moyenne-mediane

L'etape 2c a montre un ecart extreme entre la mediane (175 000 euros) et la moyenne
(2 825 906 euros) de la valeur fonciere, soit un facteur 16. Cet ecart est tire vers
le haut par les transactions extremes (la transaction a 1 milliard pese a elle seule
dans la moyenne).

La mandante demande si cet ecart se reduit en segmentant par type de bien ou par
localisation. Ce notebook examine :

1. Ratio moyenne/mediane par type de bien
2. Ratio moyenne/mediane par departement (top 20 ecarts)
3. Ratio moyenne/mediane croise type x departement (appartements)
4. Effet de la suppression des valeurs extremes sur l'ecart global

## Mode d'emploi

1. Le chemin est défini dans la **cellule 2**.
2. Executer les cellules dans l'ordre (Maj + Entree).

## Cellule 1 — Installation

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
print("Bibliotheques pretes.")

Bibliotheques pretes.


## Cellule 2 — Reglages

**Seule cellule a modifier.**

In [2]:
import duckdb
from pathlib import Path

# Chemin a renseigner
FICHIER = Path(r"./data/dvf-2022.parquet")
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pq = str(FICHIER)

assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes : {nb_lignes:,}".replace(",", " "))

Fichier : dvf-2022.parquet
Lignes : 4 617 590


## Cellule 3 — Rappel : ecart global

Le ratio moyenne/mediane global, pour reference. Un ratio de 1 signifierait une
distribution parfaitement symetrique. Plus le ratio est eleve, plus la distribution
est deformee par des valeurs extremes.

In [3]:
r = con.execute(f"""
    SELECT
        COUNT(*) AS nb,
        ROUND(MEDIAN(\"Valeur fonciere\"), 0) AS mediane,
        ROUND(AVG(\"Valeur fonciere\"), 0) AS moyenne,
        ROUND(AVG(\"Valeur fonciere\") / NULLIF(MEDIAN(\"Valeur fonciere\"), 0), 1) AS ratio
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
""").fetchone()

print("Ecart global")
print("=" * 50)
print(f"  Lignes avec prix > 0 : {r[0]:>12,}".replace(",", " "))
print(f"  Mediane              : {r[1]:>12,.0f} EUR".replace(",", " "))
print(f"  Moyenne              : {r[2]:>12,.0f} EUR".replace(",", " "))
print(f"  Ratio moyenne/mediane: {r[3]:>12.1f}")

Ecart global
  Lignes avec prix > 0 :    4 586 415
  Mediane              :      175 000 EUR
  Moyenne              :    2 825 926 EUR
  Ratio moyenne/mediane:         16.1


## Cellule 4 — Ratio moyenne/mediane par type de bien

L'ecart se reduit-il en segmentant par type de bien ? Les dependances sont
artificiellement gonflees par le piege semantique (le prix affiche est celui de
la mutation entiere). Le ratio devrait etre plus faible pour les maisons et
les appartements.

In [11]:
r = con.execute(f"""
    SELECT
        COALESCE(\"Type local\", '(vide)') AS type_local,
        COUNT(*) AS nb,
        ROUND(MEDIAN(\"Valeur fonciere\"), 0) AS mediane,
        ROUND(AVG(\"Valeur fonciere\"), 0) AS moyenne,
        ROUND(AVG(\"Valeur fonciere\") / NULLIF(MEDIAN(\"Valeur fonciere\"), 0), 1) AS ratio
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
    GROUP BY 1
    ORDER BY 5 DESC
""").fetchall()

# Ratio global
g = con.execute(f"""
    SELECT ROUND(AVG("Valeur fonciere") / NULLIF(MEDIAN("Valeur fonciere"), 0), 1)
    FROM '{pq}'
    WHERE "Valeur fonciere" > 0
""").fetchone()[0]
label_g = 'GLOBAL'
vide = ''
print(f"  {label_g:<38} {vide:>8} {vide:>10}     {vide:>12}     {g:>6.1f}")

ConnectionException: Connection Error: Connection already closed!

## Cellule 5 — Ratio moyenne/mediane par departement (top 20 ecarts)

Quels departements ont les distributions les plus deformees ?
Les departements avec des transactions tres cheres (Paris, Hauts-de-Seine)
devraient avoir un ratio plus eleve.

In [5]:
r = con.execute(f"""
    SELECT
        \"Code departement\",
        COUNT(*) AS nb,
        ROUND(MEDIAN(\"Valeur fonciere\"), 0) AS mediane,
        ROUND(AVG(\"Valeur fonciere\"), 0) AS moyenne,
        ROUND(AVG(\"Valeur fonciere\") / NULLIF(MEDIAN(\"Valeur fonciere\"), 0), 1) AS ratio
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
    GROUP BY 1
    ORDER BY 5 DESC
    LIMIT 20
""").fetchall()

print(f"{'Dept':<8} {'Nb':>10} {'Mediane':>12} {'Moyenne':>14} {'Ratio':>8}")
print("-" * 56)
for dept, nb, med, moy, ratio in r:
    print(f"  {dept:<6} {nb:>8,} {med:>10,.0f} EUR {moy:>12,.0f} EUR {ratio:>6.1f}".replace(",", " "))

Dept             Nb      Mediane        Moyenne    Ratio
--------------------------------------------------------
  56       92 544    287 000 EUR  107 071 470 EUR  373.1
  22       70 032    150 000 EUR    6 649 699 EUR   44.3
  93       59 749    284 850 EUR    2 868 114 EUR   10.1
  92       79 912    458 000 EUR    3 956 037 EUR    8.6
  973       5 569    305 000 EUR    2 314 171 EUR    7.6
  75       95 824    530 725 EUR    3 441 842 EUR    6.5
  974      23 329    215 000 EUR    1 211 161 EUR    5.6
  971       7 974    189 000 EUR    1 026 547 EUR    5.4
  972       8 026    194 078 EUR      907 099 EUR    4.7
  13      108 384    265 000 EUR    1 110 808 EUR    4.2
  94       56 282    320 000 EUR    1 322 097 EUR    4.1
  65       21 735    107 141 EUR      420 735 EUR    3.9
  78       69 468    313 200 EUR    1 185 546 EUR    3.8
  59      122 024    173 000 EUR      652 489 EUR    3.8
  90        8 554    148 310 EUR      549 357 EUR    3.7
  77       84 087    235 000 EU

## Cellule 6 — Ratio moyenne/mediane par departement (10 plus faibles ecarts)

A l'inverse, quels departements ont les distributions les plus regulieres ?

In [6]:
r = con.execute(f"""
    SELECT
        \"Code departement\",
        COUNT(*) AS nb,
        ROUND(MEDIAN(\"Valeur fonciere\"), 0) AS mediane,
        ROUND(AVG(\"Valeur fonciere\"), 0) AS moyenne,
        ROUND(AVG(\"Valeur fonciere\") / NULLIF(MEDIAN(\"Valeur fonciere\"), 0), 1) AS ratio
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
    GROUP BY 1
    ORDER BY 5 ASC
    LIMIT 10
""").fetchall()

print(f"{'Dept':<8} {'Nb':>10} {'Mediane':>12} {'Moyenne':>14} {'Ratio':>8}")
print("-" * 56)
for dept, nb, med, moy, ratio in r:
    print(f"  {dept:<6} {nb:>8,} {med:>10,.0f} EUR {moy:>12,.0f} EUR {ratio:>6.1f}".replace(",", " "))

Dept             Nb      Mediane        Moyenne    Ratio
--------------------------------------------------------
  15       15 515     84 000 EUR      118 838 EUR    1.4
  50       46 850    122 000 EUR      174 367 EUR    1.4
  27       38 786    171 500 EUR      253 843 EUR    1.5
  61       25 458    105 000 EUR      162 646 EUR    1.5
  87       39 883     95 000 EUR      146 001 EUR    1.5
  39       23 210    100 000 EUR      147 910 EUR    1.5
  02       35 150     95 000 EUR      139 849 EUR    1.5
  32       25 207    135 000 EUR      207 232 EUR    1.5
  81       33 585    136 300 EUR      201 654 EUR    1.5
  82       24 482    147 000 EUR      221 359 EUR    1.5


## Cellule 7 — Croisement type x departement pour les appartements

Pour les appartements uniquement, le ratio par departement. C'est la segmentation
la plus fine : un seul type de bien, une seule localisation. L'ecart devrait etre
nettement plus faible que le ratio global de 16.

In [7]:
r = con.execute(f"""
    SELECT
        \"Code departement\",
        COUNT(*) AS nb,
        ROUND(MEDIAN(\"Valeur fonciere\"), 0) AS mediane,
        ROUND(AVG(\"Valeur fonciere\"), 0) AS moyenne,
        ROUND(AVG(\"Valeur fonciere\") / NULLIF(MEDIAN(\"Valeur fonciere\"), 0), 1) AS ratio
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
    AND \"Type local\" = 'Appartement'
    GROUP BY 1
    HAVING COUNT(*) > 1000
    ORDER BY 5 DESC
    LIMIT 15
""").fetchall()

print("Appartements uniquement (departements avec > 1 000 transactions) :")
print(f"{'Dept':<8} {'Nb':>10} {'Mediane':>12} {'Moyenne':>14} {'Ratio':>8}")
print("-" * 56)
for dept, nb, med, moy, ratio in r:
    print(f"  {dept:<6} {nb:>8,} {med:>10,.0f} EUR {moy:>12,.0f} EUR {ratio:>6.1f}".replace(",", " "))

Appartements uniquement (departements avec > 1 000 transactions) :
Dept             Nb      Mediane        Moyenne    Ratio
--------------------------------------------------------
  974       5 360    220 250 EUR    2 493 726 EUR   11.3
  93       15 825    240 639 EUR    2 702 300 EUR   11.2
  971       2 050    190 000 EUR    1 786 688 EUR    9.4
  92       24 839    411 000 EUR    3 770 402 EUR    9.2
  65        3 601    100 000 EUR      842 258 EUR    8.4
  2B        2 184    160 650 EUR    1 199 376 EUR    7.5
  972       2 481    160 000 EUR      882 876 EUR    5.5
  05        3 802    133 783 EUR      644 562 EUR    4.8
  72        2 943    115 000 EUR      548 909 EUR    4.8
  33       14 485    216 900 EUR    1 038 292 EUR    4.8
  75       42 779    495 000 EUR    2 361 432 EUR    4.8
  13       29 131    200 000 EUR      933 549 EUR    4.7
  54        7 722    135 000 EUR      604 487 EUR    4.5
  31       16 293    165 000 EUR      616 763 EUR    3.7
  78       13 010    

## Cellule 8 — Effet de la winsorisation sur l'ecart global

Que se passe-t-il si on ecrete les valeurs extremes ? Calculer le ratio
en excluant les prix au-dessus du P99, puis du P95. Cela montre l'impact
des outliers sur la moyenne.

In [8]:
for label, seuil in [('Sans filtre', None), ('P99 (< 48M)', 48000000), ('P95 (< 1.1M)', 1100000), ('< 1M', 1000000), ('< 500k', 500000)]:
    where = f'AND \"Valeur fonciere\" < {seuil}' if seuil else ''
    r = con.execute(f"""
        SELECT
            COUNT(*),
            ROUND(MEDIAN(\"Valeur fonciere\"), 0),
            ROUND(AVG(\"Valeur fonciere\"), 0),
            ROUND(AVG(\"Valeur fonciere\") / NULLIF(MEDIAN(\"Valeur fonciere\"), 0), 1)
        FROM '{pq}'
        WHERE \"Valeur fonciere\" > 0 {where}
    """).fetchone()
    nb, med, moy, ratio = r
    print(f"  {label:<20} Nb={nb:>10,}  Med={med:>10,.0f}  Moy={moy:>10,.0f}  Ratio={ratio:.1f}".replace(",", " "))

  Sans filtre          Nb= 4 586 415  Med=   175 000  Moy= 2 825 926  Ratio=16.1
  P99 (< 48M)          Nb= 4 545 770  Med=   174 000  Moy=   494 466  Ratio=2.8
  P95 (< 1.1M)         Nb= 4 317 448  Med=   164 000  Moy=   210 228  Ratio=1.3
  < 1M                 Nb= 4 295 664  Med=   162 700  Moy=   206 020  Ratio=1.3
  < 500k               Nb= 3 950 989  Med=   150 000  Moy=   165 285  Ratio=1.1


## Cellule 9 — Synthese de l'etape 2g

In [9]:
print("Synthese — Ecart moyenne/mediane DVF 2022")
print("=" * 55)
print()
print(f"  Ratio global                           : 16.1")
print()

# Ratios par type
r = con.execute(f"""
    SELECT
        COALESCE(\"Type local\", '(vide)'),
        ROUND(AVG(\"Valeur fonciere\") / NULLIF(MEDIAN(\"Valeur fonciere\"), 0), 1)
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchall()
print("  Par type de bien :")
for typ, ratio in r:
    print(f"    {typ:<38} ratio = {ratio:.1f}")

Synthese — Ecart moyenne/mediane DVF 2022

  Ratio global                           : 16.1

  Par type de bien :
    Appartement                            ratio = 42.6
    Dépendance                             ratio = 23.2
    Local industriel. commercial ou assimilé ratio = 9.4
    (vide)                                 ratio = 5.4
    Maison                                 ratio = 3.5


## Cellule 10 — Fermeture

In [10]:
con.close()
print("Connexion DuckDB fermee.")

Connexion DuckDB fermee.
